# Sri Lankan Tea Auction Price Forecasting Showdown

**Objective:** Compare one-month-ahead forecasting accuracy of a Naive baseline, ARIMA(1,1,1), and Gradient Boosting using MAPE, MAE, and RMSE.

**Data:** Monthly average all-elevations tea auction prices in LKR/kg, January 2016–December 2022 (84 observations). Source: [Tea Exporters Association Sri Lanka — Tea Prices](https://teasrilanka.org/tea-prices). The dataset is transcribed from the source table; selected dates are checked below, but full source-by-source verification is still advisable.

**Evaluation:** Expanding-window, one-step-ahead backtesting over July 2021–December 2022 (18 months). Each prediction uses only prices observed before its target month. The models are evaluated on the same months.

## Dependencies

In [ ]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn statsmodels

In [ ]:
from pathlib import Path
import io
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from IPython.display import display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Pandas:', pd.__version__)

## Data loading

Look for `data/tea_prices_2016_2022.csv` relative to the working directory or request it via the Colab upload dialog when necessary.

In [ ]:
candidates = [
    Path('data/tea_prices_2016_2022.csv'),
    Path('../data/tea_prices_2016_2022.csv'),
    Path('/content/tea_prices_2016_2022.csv')
]
source_path = next((p for p in candidates if p.exists()), None)
if source_path is not None:
    raw = pd.read_csv(source_path)
    print('Loaded:', source_path)
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Place tea_prices_2016_2022.csv in data/ (repository root) or ../data/ (from notebooks/).') from exc
    print('Upload tea_prices_2016_2022.csv:')
    uploaded = files.upload()
    if 'tea_prices_2016_2022.csv' not in uploaded:
        raise FileNotFoundError('Please upload the provided tea_prices_2016_2022.csv file.')
    raw = pd.read_csv(io.BytesIO(uploaded['tea_prices_2016_2022.csv']))

display(raw.head())

## Data validation and preprocessing

Check monthly continuity, unique dates, numerical prices, and missing values before modelling.

In [ ]:
required = {'date', 'price_lkr_per_kg'}
assert required.issubset(raw.columns), f'Missing columns: {required - set(raw.columns)}'
df = raw[['date', 'price_lkr_per_kg']].copy()
df['date'] = pd.to_datetime(df['date'], errors='raise')
df['price_lkr_per_kg'] = pd.to_numeric(df['price_lkr_per_kg'], errors='raise')
df = df.sort_values('date').reset_index(drop=True)
assert not df['date'].duplicated().any(), 'Duplicate month(s) found'
assert df['price_lkr_per_kg'].notna().all(), 'Missing prices found'
assert (df['price_lkr_per_kg'] > 0).all(), 'Prices must be positive for MAPE'
assert df['date'].dt.day.eq(1).all(), 'Expected first day of month timestamps'
expected_dates = pd.date_range(df.date.min(), df.date.max(), freq='MS')
assert list(df.date) == list(expected_dates), 'Some months are missing or out of order'
series = df.set_index('date')['price_lkr_per_kg'].asfreq('MS')
print('Observations:', len(series))
print('Period:', series.index.min().date(), 'through', series.index.max().date())
print('Missing:', series.isna().sum())
display(series.describe().to_frame(name='LKR per kg'))
assert len(series) >= 48, 'Need enough observations for seasonal lag 12 and meaningful evaluation'

### Source value checks

Compare six observations with values reported in the original tea-price table. These checks do not independently verify all observations.

In [ ]:
checks = {'2016-01-01': 410.88, '2019-06-01': 494.48,
          '2021-12-01': 638.23, '2022-04-01': 1350.25,
          '2022-09-01': 1617.70, '2022-12-01': 1464.85}
for date, expected in checks.items():
    got = float(series.loc[date])
    assert np.isclose(got, expected, atol=0.005), f'{date}: expected {expected}, got {got}'
print('Passed', len(checks), 'source spot-checks.')

## Exploratory data analysis

Inspect trend, variability, and potential structural changes in the price series.

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
series.plot(ax=ax, marker='o', markersize=2.5)
ax.set(title='Monthly Sri Lankan Tea Auction Price — All Elevations (2016–2022)',
       xlabel='Date', ylabel='LKR per kg')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9,4))
sns.boxplot(x=series.index.month, y=series.values, ax=ax)
ax.set(xlabel='Month (1=Jan)', ylabel='LKR per kg', title='Monthly Price Distribution (all years)')
plt.tight_layout(); plt.show()

In [ ]:
# Optional stationarity diagnostic; ARIMA with d=1 can address a nonstationary level series.
adf_p = adfuller(series)[1]
print(f'ADF p-value (level prices): {adf_p:.4f}')
print('Caution: ADF results are sensitive to structural breaks and small samples.')

## Forecast evaluation period

Initial training history: January 2016–June 2021 (66 months). Evaluation period: July 2021–December 2022 (18 months). The one-step-ahead backtest incorporates each observed test-month price into the history only after that month has been forecast. Hyperparameters are fixed rather than tuned on the test set.

In [ ]:
TEST_MONTHS = 18
train, test = series.iloc[:-TEST_MONTHS], series.iloc[-TEST_MONTHS:]
print('Train:',train.index.min().date(), 'to',train.index.max().date(),len(train))
print('Test :',test.index.min().date(), 'to',test.index.max().date(),len(test))
assert train.index.max() + pd.offsets.MonthBegin(1) == test.index.min()
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(train.index, train.values, label='Training', color='steelblue')
ax.plot(test.index, test.values, label='Testing', color='darkorange')
ax.set(title='Time-ordered Train/Test Split',ylabel='LKR per kg');ax.legend();plt.show()

## Lagged features

Gradient Boosting uses lagged prices, backward-looking rolling averages, and seasonal calendar encodings. No target-month price is supplied as an input.

In [ ]:
FEATURES = ['lag_1','lag_2','lag_3','lag_6','lag_12',
            'mean_3','mean_6','month_sin','month_cos']

def make_features(history: pd.Series) -> pd.DataFrame:
    x = pd.DataFrame(index=history.index)
    for lag in [1,2,3,6,12]:
        x[f'lag_{lag}'] = history.shift(lag)
    x['mean_3'] = history.shift(1).rolling(3).mean()
    x['mean_6'] = history.shift(1).rolling(6).mean()
    x['month_sin'] = np.sin(2 * np.pi * history.index.month / 12)
    x['month_cos'] = np.cos(2 * np.pi * history.index.month / 12)
    return x

feature_preview = make_features(train).dropna()
display(feature_preview.head())
assert len(feature_preview) > 30

## Rolling-origin forecasts

Naive uses the last observed price; ARIMA(1,1,1) is refitted on available history; Gradient Boosting is refitted on lagged features constructed from available history.

In [ ]:
def rolling_predictions(series, n_test=18):
    predictions = []
    start = len(series) - n_test
    for idx in range(start, len(series)):
        history = series.iloc[:idx]
        target_date = series.index[idx]
        actual = float(series.iloc[idx])

        # Model A — one-month naive forecast
        naive_forecast = float(history.iloc[-1])

        # Model B — ARIMA using history only
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=UserWarning)
            warnings.filterwarnings('ignore', message='Non-invertible starting MA parameters')
            fitted_arima = ARIMA(history, order=(1,1,1),
                                 enforce_stationarity=False,
                                 enforce_invertibility=False).fit()
        arima_forecast = float(fitted_arima.forecast(steps=1).iloc[0])

        # Model C — Gradient Boosting trained only on past rows
        training_features = make_features(history)
        training_data = training_features.copy()
        training_data['target'] = history
        training_data = training_data.dropna()
        gb = GradientBoostingRegressor(n_estimators=100,
                                       learning_rate=0.05,
                                       max_depth=2,
                                       min_samples_leaf=3,
                                       random_state=RANDOM_STATE)
        gb.fit(training_data[FEATURES], training_data['target'])

        # Append a placeholder for the target month, so shifts refer only to history
        extended = pd.concat([history, pd.Series([np.nan], index=[target_date])])
        x_next = make_features(extended).loc[[target_date], FEATURES]
        assert x_next.notna().all().all(), 'Feature generation produced NaNs'
        gb_forecast = float(gb.predict(x_next)[0])

        predictions.append({'date':target_date, 'actual':actual,
                            'Naive':naive_forecast, 'ARIMA':arima_forecast,
                            'Gradient Boosting':gb_forecast})
    return pd.DataFrame(predictions).set_index('date')

results = rolling_predictions(series, TEST_MONTHS)
display(results.round(2))

## Forecast error metrics

MAPE (%) is the primary metric. MAE and RMSE (LKR/kg) are supporting metrics. MAPE requires positive actual values.

In [ ]:
def calculate_metrics(actual, predicted):
    a, p = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    assert len(a) == len(p) and np.all(np.isfinite(a)) and np.all(np.isfinite(p))
    if np.any(a == 0): raise ValueError('MAPE undefined when actual price is zero')
    return {
        'MAPE (%)': float(np.mean(np.abs((a-p)/a))*100),
        'MAE (LKR/kg)': float(mean_absolute_error(a,p)),
        'RMSE (LKR/kg)': float(np.sqrt(mean_squared_error(a,p)))
    }
metrics = pd.DataFrame({model: calculate_metrics(results['actual'],results[model])
                        for model in ['Naive','ARIMA','Gradient Boosting']}).T
metrics = metrics.sort_values('MAPE (%)')
display(metrics.round(3))
print('Best model by test MAPE:',metrics.index[0])

## Actual versus predicted prices

In [ ]:
fig,ax=plt.subplots(figsize=(13,5))
results['actual'].plot(ax=ax, marker='o', linewidth=2.5, label='Actual', color='black')
for col in ['Naive','ARIMA','Gradient Boosting']:
    results[col].plot(ax=ax, marker='.', alpha=0.85, label=col)
ax.set(title='One-Month-Ahead Forecasts vs Actual Tea Prices',ylabel='LKR per kg',xlabel='Month')
ax.legend();plt.tight_layout();plt.show()

ax=metrics['MAPE (%)'].sort_values().plot.bar(figsize=(8,4))
ax.set(title='Test MAPE — Lower is Better',ylabel='MAPE (%)',xlabel='Model')
plt.xticks(rotation=20);plt.tight_layout();plt.show()

## Error analysis

Examine the 10 largest absolute errors for each model and their residuals. Large errors indicate months where the observed price changed substantially relative to model expectations.

In [ ]:
all_errors = []
for model in ['Naive','ARIMA','Gradient Boosting']:
    e = pd.DataFrame({'actual':results['actual'], 'predicted':results[model]})
    e['model'] = model
    e['error_lkr_per_kg'] = e['actual']-e['predicted']
    e['absolute_error'] = e['error_lkr_per_kg'].abs()
    e['absolute_pct_error'] = 100*e['absolute_error']/e['actual']
    e = e.sort_values('absolute_error',ascending=False).head(10)
    all_errors.append(e.reset_index())
worst_errors=pd.concat(all_errors,ignore_index=True)
for model in ['Naive','ARIMA','Gradient Boosting']:
    print('\n--- Top 10 errors:', model,'---')
    display(worst_errors[worst_errors.model==model].round(2))

In [ ]:
fig, axes = plt.subplots(3,1, figsize=(12,9), sharex=True)
for ax, model in zip(axes,['Naive','ARIMA','Gradient Boosting']):
    residuals=results['actual']-results[model]
    residuals.plot(ax=ax,marker='o',label='Actual − Forecast')
    ax.axhline(0,color='black',linewidth=1)
    ax.set(ylabel='Error (LKR/kg)',title=f'{model}: residuals')
plt.tight_layout();plt.show()

print('Interpretation prompts:')
print('1. Which months have the largest absolute errors?')
print('2. Did the sharp price changes around 2022 increase errors?')
print('3. Which model responded fastest to rapid changes?')
print('4. Does the lowest MAPE model also have the lowest RMSE?')
print('5. Can we generalize from one short 18-month test period?')

## Export results

Write evaluation tables and forecasting visualization to `results/` relative to the current working directory.

In [ ]:
output_dir=Path('results')
output_dir.mkdir(parents=True,exist_ok=True)
results.to_csv(output_dir/'forecast_comparison.csv')
metrics.to_csv(output_dir/'model_performance.csv',index_label='model')
worst_errors.to_csv(output_dir/'top_10_errors_per_model.csv',index=False)
fig,ax=plt.subplots(figsize=(12,5))
results.plot(ax=ax)
ax.set(title='Tea Price Forecasts and Actual Values',ylabel='LKR/kg')
fig.tight_layout();fig.savefig(output_dir/'forecast_chart.png',dpi=160)
plt.show()
print('Saved files:',*[p.name for p in output_dir.iterdir()],sep='\n  - ')